# Módulo 2 - Manipulação de Dados com Pandas

Este notebook foi organizado para servir como material introdutório de apoio ao **Módulo 2: Manipulação de Dados com Pandas**.

Um slide com conteúdo complementar do módulo 2 também está disponível na pasta **slides**.

Ao longo do material, vamos usar colunas do dataset, como:

- `CITY`, `STATE`, `CAPITAL`
- `IBGE_RES_POP`, `ESTIMATED_POP`
- `AREA`, `GDP`, `GDP_CAPITA`
- `IDHM`, `IDHM_Educacao`, `IDHM_Renda`
- `HOTELS`, `BEDS`, `Cars`, `Motorcycles`

Ao final deste material, você deverá ser capaz de:

- entender o que são **Series** e **DataFrames**;
- carregar dados de diferentes fontes;
- inspecionar a estrutura de um conjunto de dados;
- identificar e tratar valores ausentes;
- remover duplicatas e ajustar tipos de dados;
- filtrar, ordenar e transformar informações;
- agrupar dados para responder perguntas analíticas;
- calcular medidas estatísticas descritivas básicas.

> **Importante:** como este é um módulo introdutório, nem toda análise aqui busca rigor estatístico avançado. O objetivo é ensinar o fluxo de trabalho com dados reais, usando um dataset rico e próximo da realidade brasileira.


## Por que o Pandas é tão importante?

Em Ciência de Dados, raramente recebemos dados “prontos”. Em geral, as bases vêm com:
- colunas demais;
- nomes pouco intuitivos;
- valores ausentes;
- duplicidades;
- tipos incorretos;
- informações vindas de várias fontes.

O **Brazilian Cities Dataset** é um ótimo exemplo disso: ele reúne dezenas de variáveis públicas sobre os municípios brasileiros, cobrindo população, economia, infraestrutura, geografia e desenvolvimento humano. Por isso, ele é excelente para aprender **manipulação de dados reais**.

## Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 80)

## Estruturas fundamentais do Pandas

As duas estruturas principais do Pandas são:

### 1. Series
Uma **Series** é uma estrutura unidimensional. Podemos pensar nela como uma única coluna de dados.

### 2. DataFrame
Um **DataFrame** é uma estrutura bidimensional, semelhante a uma tabela ou planilha, com linhas e colunas.

No contexto do dataset de municípios, uma coluna como `GDP` pode ser vista como uma **Series**, enquanto a tabela inteira com todas as cidades é um **DataFrame**.

## Exemplo de Series com uma variável do contexto do dataset

In [ ]:
idhm_series = pd.Series(
    [0.754, 0.682, 0.746, 0.778, 0.751],
    index=["Fortaleza", "Caucaia", "Recife", "Curitiba", "Florianópolis"],
    name="IDHM"
)
idhm_series

Nesse exemplo, cada cidade está associada a um valor de **IDHM**.  
Temos, portanto, uma única dimensão de dados: uma cidade e seu respectivo índice.

In [ ]:
print("Primeiro valor:", idhm_series.iloc[0])
print("Cidade na primeira posição:", idhm_series.index[0])
print("Média do IDHM nesse exemplo:", idhm_series.mean())

## Exemplo de DataFrame com colunas do Brazilian Cities

In [ ]:
dados_exemplo = {
    "CITY": ["Fortaleza", "Caucaia", "Recife", "Curitiba", "Fortaleza"],
    "STATE": ["CE", "CE", "PE", "PR", "CE"],
    "CAPITAL": [1, 0, 1, 1, 1],
    "IBGE_RES_POP": [2452185, 325441, 1537704, 1933105, 2452185],
    "AREA": [314.93, 1228.51, 218.84, 434.97, 314.93],
    "GDP": ["65151000", "9500000", "52017000", "98000000", "65151000"],
    "IDHM": [0.754, 0.682, 0.772, np.nan, 0.754],
    "HOTELS": [120, np.nan, 95, 160, 120]
}

df_exemplo = pd.DataFrame(dados_exemplo)
df_exemplo

Agora temos um **DataFrame**, isto é, uma tabela com múltiplas colunas representando características de municípios.

Observe que esse exemplo já foi montado com alguns problemas comuns de bases reais:
- uma linha duplicada;
- valores ausentes;
- uma coluna numérica (`GDP`) armazenada como texto.

Esses “problemas propositais” ajudam a demonstrar as etapas de limpeza que veremos depois.

## Leitura de dados: CSV, Excel e JSON

Uma das maiores vantagens do Pandas é a facilidade para ler diferentes formatos de arquivo.  
Em projetos reais, é comum receber dados em:

- **CSV**: formato tabular simples e muito usado em análise de dados;
- **Excel**: planilhas `.xlsx`;
- **JSON**: formato muito comum em APIs e sistemas web.

### Como ler o dataset em um projeto real

Quando o arquivo do dataset estiver disponível no Colab ou na mesma pasta do notebook, a leitura pode ser feita assim:

```python
df = pd.read_csv("brazilian_cities.csv")
```

Também é possível adaptar para outros formatos:

```python
df_excel = pd.read_excel("brazilian_cities.xlsx")
df_json = pd.read_json("brazilian_cities.json")
```

No bloco abaixo, deixamos um carregamento **robusto**: ele tenta localizar automaticamente um arquivo CSV do dataset.  
Se o arquivo não for encontrado, o notebook usa uma **amostra didática** para que o material continue funcionando.

In [ ]:
possiveis_arquivos = [
    Path("brazilian_cities.csv"),
    Path("Brazilian_Cities.csv"),
    Path("datasets/brazilian_cities.csv"),
    Path("../datasets/brazilian_cities.csv"),
    Path("/mnt/data/brazilian_cities.csv"),
]

arquivo_encontrado = next((p for p in possiveis_arquivos if p.exists()), None)

if arquivo_encontrado:
    df = pd.read_csv(arquivo_encontrado)
    print(f"Dataset carregado com sucesso: {arquivo_encontrado}")
else:
    # Amostra baseada em um recorte do Brazilian Cities Dataset
    from io import StringIO

    csv_amostra = StringIO("""
"CITY","STATE","CAPITAL","IBGE_RES_POP","IDHM","AREA","ESTIMATED_POP","RURAL_URBAN","GDP","GDP_CAPITA","GVA_MAIN","MUN_EXPENDIT","COMP_TOT","HOTELS","BEDS","Pr_Assets","Pu_Assets","Cars","Motorcycles","UBER","MAC","WAL-MART","POST_OFFICES"
"Abadia De Goiás","GO","0","6876","0.7080","147.2560","8583","Urbano","166412","20665.0000","Demais serviços","28227690","284","0","0","0","0","2158","1246","0","0","0","1"
"Abadia Dos Dourados","MG","0","6704","0.6890","881.0640","6972","Rural Adjacente","180089","25592.0000","Demais serviços","17909274","476","0","0","0","0","2227","1142","0","0","0","1"
"Abadiânia","GO","0","15757","0.6890","1045.1270","19614","Rural Adjacente","287984","15628.0000","Demais serviços","37513019","288","1","34","33724584","67091904","2838","1426","0","0","0","3"
"Abaetetuba","PA","0","141100","0.6280","1610.6510","156292","Urbano","1249255","8222.0000","Administração, defesa, educação e saúde públicas e seguridade social","0","931","2","4","76181384","800078483","5277","25661","0","0","0","2"
"Abaeté","MG","0","22690","0.6980","1817.0670","23223","Urbano","430235","18250.0000","Demais serviços","0","621","2","2","44974716","371922572","6928","2953","0","0","0","4"
"Abaiara","CE","0","10496","0.6280","180.0800","11663","Rural Adjacente","73151","6370.0000","Administração, defesa, educação e saúde públicas e seguridade social","0","86","0","0","0","0","553","1674","0","0","0","1"
"Abaré","BA","0","17064","0.5750","1604.9230","19814","Rural Remoto","124754","6257.0000","Administração, defesa, educação e saúde públicas e seguridade social","0","87","1","0","21823314","0","613","1532","0","0","0","1"
"Abatiá","PR","0","7764","0.6870","228.7170","7507","Rural Adjacente","165048","21174.0000","Agricultura, inclusive apoio à agricultura e a pós colheita","0","285","0","1","0","45976288","2168","912","0","0","0","1"
"Abaíra","BA","0","8316","0.6030","538.6770","8767","Rural Remoto","64325","6983.0000","Administração, defesa, educação e saúde públicas e seguridade social","0","191","0","0","0","0","896","696","0","0","0","1"
""")
    df = pd.read_csv(csv_amostra)
    print("Arquivo CSV não encontrado. Foi carregada uma amostra didática inspirada no dataset.")

print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")


## Primeira leitura da base

Assim que carregamos um dataset, as primeiras perguntas devem ser:

- quantas linhas e colunas existem?
- quais variáveis estão disponíveis?
- os nomes das colunas fazem sentido?
- os dados parecem consistentes à primeira vista?

Essas perguntas orientam a etapa de **exploração inicial**, antes de qualquer limpeza mais profunda.


In [ ]:
df.head(10)

## Entendendo o significado de algumas colunas do Brazilian Cities

O dataset reúne indicadores de áreas diferentes. Isso é ótimo para análise, mas exige atenção redobrada ao interpretar cada variável.

### Exemplos de grupos de colunas

- **Identificação e localização:** `CITY`, `STATE`, `CAPITAL`
- **População:** `IBGE_RES_POP`, `ESTIMATED_POP`
- **Território:** `AREA`
- **Desenvolvimento humano:** `IDHM`, `IDHM_Renda`, `IDHM_Educacao`
- **Economia:** `GDP`, `GDP_CAPITA`, `GVA_MAIN`
- **Infraestrutura e serviços:** `HOTELS`, `BEDS`, `Cars`, `Motorcycles`, `POST_OFFICES`
- **Tipologia territorial:** `RURAL_URBAN`

Esse conhecimento é importante porque **o tratamento depende do significado da coluna**.  
Por exemplo, preencher `HOTELS` com zero pode fazer sentido em alguns contextos, mas preencher `IDHM` com zero quase sempre seria inadequado.


In [ ]:
colunas_exemplo = [col for col in [
    "CITY", "STATE", "CAPITAL", "IBGE_RES_POP", "ESTIMATED_POP",
    "AREA", "IDHM", "GDP", "GDP_CAPITA", "GVA_MAIN",
    "HOTELS", "BEDS", "Cars", "Motorcycles", "POST_OFFICES"
] if col in df.columns]

df[colunas_exemplo].head(10)

## Exploração inicial dos dados

Antes de limpar, transformar ou visualizar a base, precisamos **entender o que temos em mãos**.  
Essa etapa costuma ser chamada de **exploração inicial** e funciona como uma leitura diagnóstica do dataset.

Aqui, buscamos responder perguntas como:

- quantas linhas e colunas existem?
- quais são os nomes das colunas?
- quais variáveis parecem numéricas e quais parecem textuais?
- existem valores ausentes?
- há colunas com tipo inadequado?
- os valores parecem coerentes com o contexto?

Essa etapa é importante porque evita erros mais à frente.  
Por exemplo: se uma coluna que deveria ser numérica foi lida como texto, cálculos e ordenações podem gerar resultados incorretos.

### Comandos muito usados nessa fase

- `head()` → mostra as primeiras linhas;
- `shape` → informa quantas linhas e colunas existem;
- `columns` → lista os nomes das colunas;
- `info()` → resume tipos de dados e valores não nulos;
- `describe()` → gera estatísticas descritivas das colunas.

Começamos olhando a base de forma geral e, em seguida, veremos um exemplo menor e mais simples para reforçar a lógica.


In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

### Exemplo simples de exploração inicial

Antes de seguir com a base completa, vale observar um exemplo menor.  
Com poucas linhas, fica mais fácil perceber o papel de cada comando.


In [ ]:
df_exemplo_inicial = pd.DataFrame({
    "nome": ["Ana", "Bruno", "Carla"],
    "idade": [20, 22, 19],
    "cidade": ["Fortaleza", "Caucaia", "Maracanaú"]
})

print("Dimensão do DataFrame:", df_exemplo_inicial.shape)
print("\nColunas:")
print(df_exemplo_inicial.columns.tolist())

print("\nPrimeiras linhas:")
display(df_exemplo_inicial.head())

print("\nResumo das informações:")
df_exemplo_inicial.info()


Nesse exemplo pequeno, já conseguimos enxergar a lógica da exploração inicial:

- `shape` mostra o tamanho da tabela;
- `columns.tolist()` lista as colunas;
- `head()` exibe uma amostra rápida;
- `info()` ajuda a verificar tipos e preenchimento dos dados.

Em bases grandes, fazemos exatamente a mesma coisa — apenas com muito mais colunas e linhas.


## Selecionando uma amostra de colunas para estudo

Datasets reais podem ter muitas variáveis. Em vez de tentar entender tudo ao mesmo tempo, é útil separar um subconjunto de colunas mais relevantes para a aula.

Abaixo, montamos uma visão enxuta com variáveis de identificação, população, economia, desenvolvimento e infraestrutura.


In [ ]:
colunas_estudo = [col for col in [
    "CITY", "STATE", "CAPITAL", "IBGE_RES_POP", "ESTIMATED_POP", "AREA",
    "IDHM", "GDP", "GDP_CAPITA", "GVA_MAIN", "HOTELS", "BEDS",
    "Cars", "Motorcycles", "POST_OFFICES"
] if col in df.columns]

df_base = df[colunas_estudo].copy()
df_base.head(10)

Criar subconjuntos como `df_base` ajuda a reduzir a complexidade inicial do problema.  
Isso é especialmente útil em aulas introdutórias, quando ainda estamos aprendendo a explorar os dados com segurança.


No dataset de municípios, essa inspeção inicial é especialmente importante porque temos colunas de naturezas muito diferentes.  
Algumas medem população, outras economia, outras infraestrutura e outras desenvolvimento humano. Além disso, nem todas necessariamente estarão preenchidas em todos os municípios.

## Identificando valores ausentes (Nulos / NaN)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

Valores ausentes são comuns em bases reais. Eles podem aparecer por:
- falhas de coleta;
- indisponibilidade da informação para determinado município;
- erro de integração entre fontes;
- ausência real do fenômeno medido.

Por exemplo, colunas como `HOTELS`, `BEDS`, `GDP_CAPITA` ou `IDHM_Educacao` podem apresentar lacunas em algumas cidades.

## Tratando valores ausentes

In [ ]:
# Cria uma cópia do DataFrame para não alterar o original
df_tratado = df_base.copy() if 'df_base' in globals() else df.copy()

# Lista com todas as colunas numéricas do dataset
colunas_numericas = [
    "CAPITAL", "IBGE_RES_POP", "IBGE_RES_POP_BRAS", "IBGE_RES_POP_ESTR",
    "IBGE_DU", "IBGE_DU_URBAN", "IBGE_DU_RURAL", "IBGE_POP",
    "IBGE_1", "IBGE_1-4", "IBGE_5-9", "IBGE_10-14", "IBGE_15-59", "IBGE_60+",
    "IBGE_PLANTED_AREA", "IBGE_CROP_PRODUCTION_$",
    "IDHM Ranking 2010", "IDHM", "IDHM_Renda", "IDHM_Longevidade", "IDHM_Educacao",
    "LONG", "LAT", "ALT", "PAY_TV", "FIXED_PHONES", "AREA",
    "ESTIMATED_POP", "GVA_AGROPEC", "GVA_INDUSTRY", "GVA_SERVICES", "GVA_PUBLIC",
    "GVA_TOTAL", "TAXES", "GDP", "POP_GDP", "GDP_CAPITA", "MUN_EXPENDIT",
    "COMP_TOT", "COMP_A", "COMP_B", "COMP_C", "COMP_D", "COMP_E", "COMP_F",
    "COMP_G", "COMP_H", "COMP_I", "COMP_J", "COMP_K", "COMP_L", "COMP_M",
    "COMP_N", "COMP_O", "COMP_P", "COMP_Q", "COMP_R", "COMP_S", "COMP_T", "COMP_U",
    "HOTELS", "BEDS", "Pr_Agencies", "Pu_Agencies", "Pr_Bank", "Pu_Bank",
    "Pr_Assets", "Pu_Assets", "Cars", "Motorcycles", "Wheeled_tractor",
    "UBER", "MAC", "WAL-MART", "POST_OFFICES"
]

# Lista com colunas categóricas
colunas_categoricas = [
    "CITY", "STATE", "RURAL_URBAN", "REGIAO_TUR", "CATEGORIA_TUR", "GVA_MAIN"
]

# 1. Converter colunas numéricas
for col in colunas_numericas:
    if col in df_tratado.columns:
        df_tratado[col] = pd.to_numeric(df_tratado[col], errors="coerce")

# 2. Tratar valores inválidos "0" e 0 em colunas categóricas
for col in colunas_categoricas:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].replace(["0", 0], pd.NA)

# 3. Tratar ausentes em colunas específicas

# Nessas colunas, faz sentido substituir ausentes por 0
colunas_zero = [
    "HOTELS", "BEDS", "POST_OFFICES", "PAY_TV", "FIXED_PHONES",
    "Pr_Agencies", "Pu_Agencies", "Pr_Bank", "Pu_Bank",
    "Cars", "Motorcycles", "Wheeled_tractor", "MAC", "WAL-MART", "UBER"
]

for col in colunas_zero:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].fillna(0)

# Nessas colunas, usamos a mediana porque são quantitativas
colunas_mediana = [
    "IDHM", "IDHM_Renda", "IDHM_Longevidade", "IDHM_Educacao",
    "GDP", "GDP_CAPITA", "ESTIMATED_POP", "AREA", "MUN_EXPENDIT"
]

for col in colunas_mediana:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].fillna(df_tratado[col].median())

# Nas colunas categóricas, usamos o valor mais frequente
for col in colunas_categoricas:
    if col in df_tratado.columns:
        if df_tratado[col].isna().sum() > 0:
            df_tratado[col] = df_tratado[col].fillna(df_tratado[col].mode()[0])

# 4. Verificar quantos valores ausentes restaram
valores_ausentes = df_tratado.isna().sum()
valores_ausentes = valores_ausentes[valores_ausentes > 0]

print("Colunas que ainda possuem valores ausentes:")
print(valores_ausentes)

# Visualiza parte do resultado
df_tratado.head(10)

Aqui usamos estratégias diferentes conforme o contexto:
- em colunas como `HOTELS` e `BEDS`, preencher com **0** pode fazer sentido quando a ausência pode representar inexistência registrada;
- em colunas como `IDHM` ou `GDP_CAPITA`, uma estratégia simples é preencher com a **mediana**, que é menos sensível a valores extremos do que a média.

Em bases reais, a melhor escolha depende sempre do significado da variável.

## Verificando e removendo duplicatas

In [ ]:
print("Duplicatas exatas no recorte atual:", df_tratado.duplicated().sum())
print("Duplicatas por município (CITY + STATE):", df_tratado.duplicated(subset=["CITY", "STATE"]).sum())

In [ ]:
df_tratado[df_tratado.duplicated(subset=["CITY", "STATE"], keep=False)].sort_values(["STATE", "CITY"])

In [ ]:
df_tratado = df_tratado.drop_duplicates(subset=["CITY", "STATE"])
df_tratado.shape

Duplicatas distorcem resultados porque um mesmo município pode passar a ser contado mais de uma vez.  
Em análises agregadas, isso afeta médias, somas, totais e rankings.

Em muitos casos, também podemos verificar duplicidade usando colunas-chave, como `CITY` e `STATE`.

In [ ]:
df_tratado.duplicated(subset=["CITY", "STATE"]).sum()

## Verificando e corrigindo tipos de dados

In [ ]:
df_tratado.dtypes

Note que, em muitos datasets reais, algumas colunas numéricas chegam como texto.  
Isso acontece por vários motivos: presença de separadores, leitura incorreta do arquivo ou problemas de padronização.

No nosso exemplo, `GDP` foi propositalmente armazenada como texto para mostrar como corrigir esse problema.

No recorte do Brazilian Cities, por exemplo, aparecem muitas colunas quantitativas relacionadas a:
- população;
- PIB;
- infraestrutura;
- veículos;
- ativos bancários.

Essas variáveis precisam estar em formato numérico para que possamos calcular médias, medianas, rankings e indicadores derivados.


In [ ]:
# Conferindo os tipos depois das conversões
df_tratado.dtypes

## Checagens rápidas de consistência

Antes de partir para filtros e análises, vale fazer algumas verificações simples:

- existem valores negativos em colunas que não deveriam ser negativas?
- há municípios com área igual a zero?
- existem valores extremos que merecem investigação?

Essas checagens não resolvem tudo, mas ajudam a identificar problemas óbvios cedo.


In [ ]:
# Exemplos de checagens simples
problemas = {}

for coluna in ["AREA", "ESTIMATED_POP", "GDP", "HOTELS", "Cars", "Motorcycles"]:
    if coluna in df_tratado.columns:
        problemas[coluna] = {
            "negativos": int((df_tratado[coluna] < 0).sum()),
            "zeros": int((df_tratado[coluna] == 0).sum()),
            "nulos": int(df_tratado[coluna].isna().sum())
        }

pd.DataFrame(problemas).T

## Filtros e seleção de dados

Depois de conhecer a base, o próximo passo é **recortar apenas o que interessa**.  
Em bases reais, raramente usamos todas as linhas e todas as colunas ao mesmo tempo.

Os filtros ajudam a responder perguntas específicas, como:

- quais municípios têm população acima de determinado valor?
- quais capitais possuem IDHM alto?
- quais cidades de um estado específico quero analisar?
- quais colunas são realmente necessárias para minha análise?

No Pandas, isso aparece com frequência por meio de:

- seleção de colunas com `df[["coluna1", "coluna2"]]`;
- filtros condicionais com `loc`;
- combinação de condições com `&` (e) e `|` (ou);
- criação de subconjuntos menores para análise.

A ideia central é simples: **selecionar apenas o pedaço da base que faz sentido para a pergunta que estamos tentando responder**.

A seguir, veremos exemplos com o dataset e depois um exemplo menor, mais simples, para fixar o raciocínio.


In [ ]:
# Municípios com população estimada acima de 100 mil
df_tratado.loc[df_tratado["ESTIMATED_POP"] > 100_000, ["CITY", "STATE", "ESTIMATED_POP", "GDP"]]

In [ ]:
# Selecionando apenas algumas colunas de interesse
colunas_analise = [col for col in ["CITY", "STATE", "ESTIMATED_POP", "GDP", "GDP_CAPITA", "IDHM", "GVA_MAIN"] if col in df_tratado.columns]
df_tratado[colunas_analise].head(10)

In [ ]:
# Municípios capitais com IDHM acima de 0.75
df_tratado.loc[(df_tratado["CAPITAL"] == 1) & (df_tratado["IDHM"] > 0.75), ["CITY", "STATE", "CAPITAL", "IDHM"]]

### Exemplo simples de filtros e seleção

Agora vamos usar um exemplo pequeno para enxergar a mecânica dos filtros com mais clareza.


In [ ]:
df_alunos = pd.DataFrame({
    "nome": ["Ana", "Bruno", "Carla", "Diego"],
    "nota": [8.5, 6.0, 9.2, 5.5],
    "faltas": [2, 7, 1, 10]
})

# Seleção de colunas
display(df_alunos[["nome", "nota"]])

# Filtro simples
display(df_alunos.loc[df_alunos["nota"] >= 7])

# Filtro com duas condições
display(df_alunos.loc[(df_alunos["nota"] >= 7) & (df_alunos["faltas"] <= 2)])


Perceba como os filtros ficam mais intuitivos quando pensamos em perguntas concretas:

- quem são os alunos e suas notas?
- quais tiveram nota maior ou igual a 7?
- quais tiveram boa nota e poucas faltas?

Com o dataset de municípios, fazemos a mesma lógica, apenas trocando o contexto.


Filtros ajudam a responder perguntas analíticas específicas, como:
- quais municípios têm maior população?
- quais capitais apresentam melhor IDHM?
- quais cidades possuem maior PIB?

## Ordenação

Ordenar dados é útil quando queremos destacar extremos e comparar valores com mais facilidade.

Em análise de dados, ordenação aparece o tempo todo para responder perguntas como:

- quais municípios têm maior PIB?
- quais cidades têm maior IDHM?
- quais registros possuem os menores ou maiores valores de uma variável?

No Pandas, o comando mais comum é `sort_values()`, que permite:

- ordenar de forma crescente ou decrescente;
- ordenar por uma única coluna;
- ordenar por mais de uma coluna ao mesmo tempo.

Isso é importante porque muitas análises exploratórias começam justamente identificando **topos, fundos e padrões gerais** da base.

Primeiro veremos exemplos com o dataset real e depois um exemplo bem simples para visualizar o efeito da ordenação.


In [ ]:
df_tratado.sort_values("GDP", ascending=False)[["CITY", "STATE", "GDP"]].head(10)

In [ ]:
df_tratado.sort_values("IDHM", ascending=False)[["CITY", "STATE", "IDHM"]].head(10)

In [ ]:
df_tratado.sort_values("AREA", ascending=False)[["CITY", "STATE", "AREA"]].head(10)

### Exemplo simples de ordenação

Com um conjunto pequeno de dados, fica fácil visualizar a diferença entre ordem crescente e decrescente.


In [ ]:
df_vendas = pd.DataFrame({
    "produto": ["Caderno", "Caneta", "Mochila", "Livro"],
    "preco": [25.0, 3.5, 120.0, 45.0],
    "estoque": [40, 300, 15, 60]
})

print("Ordenando por preço (crescente):")
display(df_vendas.sort_values("preco"))

print("Ordenando por estoque (decrescente):")
display(df_vendas.sort_values("estoque", ascending=False))


Ao ordenar, não estamos mudando o conteúdo da base, mas sim a forma como os registros são apresentados.

Isso ajuda muito a identificar rapidamente:

- maiores e menores valores;
- itens mais caros ou mais baratos;
- registros com maior ou menor frequência.


Ordenar os dados ajuda a construir rankings e enxergar padrões com mais clareza.  
No caso do Brazilian Cities, isso é útil para comparar municípios por população, área, PIB ou indicadores sociais.

## Criação de novas colunas

In [ ]:
df_tratado["DENSIDADE_DEMOGRAFICA"] = df_tratado["ESTIMATED_POP"] / df_tratado["AREA"]
df_tratado["RAZAO_CARROS_POR_HAB"] = df_tratado["Cars"] / df_tratado["ESTIMATED_POP"]

df_tratado["FAIXA_POPULACIONAL"] = pd.cut(
    df_tratado["ESTIMATED_POP"],
    bins=[0, 100_000, 500_000, 1_000_000, np.inf],
    labels=["Até 100 mil", "100 mil a 500 mil", "500 mil a 1 milhão", "Acima de 1 milhão"]
)

df_tratado[["CITY", "STATE", "DENSIDADE_DEMOGRAFICA", "RAZAO_CARROS_POR_HAB", "FAIXA_POPULACIONAL"]].head(10)

Criar novas colunas é uma forma de gerar indicadores mais úteis para análise.

Neste exemplo:
- **densidade demográfica** relaciona população e área;
- **razão de carros por habitante** cria uma métrica proporcional;
- **faixa populacional** transforma uma variável numérica contínua em categorias interpretáveis.

## Agrupamentos e agregações

Uma parte muito importante da análise de dados é **resumir informações por grupos**.

Em vez de olhar município por município, podemos fazer perguntas em nível mais alto, como:

- qual é a média de população por estado?
- qual é o PIB total por estado?
- quantos municípios existem em cada grupo?
- como o IDHM médio varia entre categorias populacionais?

No Pandas, isso normalmente é feito com:

- `groupby()` para definir os grupos;
- funções como `mean()`, `sum()`, `count()`, `max()` e `min()` para resumir;
- `agg()` quando queremos combinar várias agregações ao mesmo tempo.

A lógica é: **separar a base em grupos e calcular um resumo para cada grupo**.

A seguir, começamos com exemplos do dataset e depois reforçamos a ideia com um exemplo pequeno e mais intuitivo.


In [ ]:
df_tratado.groupby("STATE")[["ESTIMATED_POP", "GDP", "HOTELS"]].mean(numeric_only=True).sort_values("ESTIMATED_POP", ascending=False)

In [ ]:
df_tratado.groupby("STATE").agg({
    "CITY": "count",
    "ESTIMATED_POP": "sum",
    "GDP": "mean",
    "IDHM": "mean",
    "HOTELS": "sum"
}).rename(columns={"CITY": "QTD_MUNICIPIOS"}).sort_values("ESTIMATED_POP", ascending=False)

In [ ]:
df_tratado.groupby("FAIXA_POPULACIONAL").agg({
    "GDP": ["mean", "max"],
    "IDHM": "mean",
    "HOTELS": "sum"
})

### Exemplo simples de agrupamentos e agregações

Vamos usar uma base pequena para observar a lógica do `groupby()` de forma mais direta.


In [ ]:
df_turmas = pd.DataFrame({
    "turma": ["A", "A", "B", "B", "B"],
    "aluno": ["Ana", "Bruno", "Carla", "Diego", "Eva"],
    "nota": [8.0, 7.0, 9.0, 6.5, 8.5]
})

print("Média da nota por turma:")
display(df_turmas.groupby("turma")["nota"].mean())

print("\nResumo com múltiplas agregações por turma:")
display(
    df_turmas.groupby("turma").agg({
        "aluno": "count",
        "nota": ["mean", "max", "min"]
    })
)


Esse exemplo mostra a lógica central dos agrupamentos:

1. escolhemos a coluna que define os grupos (`turma`);
2. selecionamos a variável a resumir (`nota`);
3. aplicamos uma agregação, como média, máximo, mínimo ou contagem.

No dataset de municípios, trocamos `turma` por `STATE` ou `FAIXA_POPULACIONAL`, mas o raciocínio é o mesmo.


O método `groupby()` é um dos recursos mais poderosos do Pandas porque permite responder perguntas do tipo:
- qual é a média do PIB por estado?
- quantos municípios existem em cada grupo?
- qual faixa populacional concentra mais hotéis?

## Estatística descritiva básica

In [ ]:
coluna = "IDHM"

print("Média:", df_tratado[coluna].mean())
print("Mediana:", df_tratado[coluna].median())
print("Moda:")
print(df_tratado[coluna].mode())
print("Variância:", df_tratado[coluna].var())
print("Desvio padrão:", df_tratado[coluna].std())

Essas medidas ajudam a resumir uma variável numérica:
- **média**: valor médio geral;
- **mediana**: valor central;
- **moda**: valor mais frequente;
- **variância** e **desvio padrão**: indicam o grau de dispersão.

No contexto do dataset, elas podem ser aplicadas a colunas como `GDP`, `GDP_CAPITA`, `AREA`, `ESTIMATED_POP` e `IDHM`.

## Perguntas analíticas que podem ser exploradas depois

Depois das etapas iniciais de limpeza e transformação, este dataset permite muitas investigações interessantes, por exemplo:

1. Quais estados concentram municípios com maior PIB médio?
2. Existe relação entre `IDHM` e `GDP_CAPITA`?
3. Municípios mais populosos tendem a ter mais hotéis ou mais veículos?
4. Como a tipologia `RURAL_URBAN` aparece em diferentes estados?
5. Quais atividades econômicas principais aparecem com mais frequência?

Essas perguntas podem servir como ponte para os próximos módulos, especialmente visualização de dados e análise exploratória.


## Lista de exercícios com o Dataset

1. Liste os 10 municípios com maior `GDP_CAPITA`.
2. Filtre apenas as capitais e compare `IDHM` e `GDP`.
3. Crie uma coluna de **motos por habitante**.
4. Agrupe por `STATE` e descubra quantos municípios existem em cada estado no recorte analisado.
5. Agrupe os dados por estado `STATE` e calcule o PIB total (`GDP`) por estado.
6. Verifique quais municípios têm `HOTELS` maior que zero e `UBER` igual a zero.
7. Ordene os municípios por `ESTIMATED_POP` e observe os maiores valores.
8. Compare a média de `IDHM` por estado.
9. Calcule o PIB médio `GDP` por estado e ordene do maior para o menor. 
10. Exiba qual estado possui a maior média de `IDHM`. 


## Conclusão

Neste notebook, usamos o **Brazilian Cities Dataset** para praticar o fluxo essencial de manipulação de dados com Pandas:

- leitura de arquivos;
- inspeção da estrutura;
- identificação de valores ausentes;
- remoção de duplicatas;
- correção de tipos;
- filtros, ordenação e criação de colunas;
- agrupamentos e estatística descritiva.

Mais importante do que decorar comandos é entender o raciocínio por trás de cada etapa.  
Em Ciência de Dados, a qualidade da análise depende fortemente da qualidade da preparação dos dados.

Nos próximos tópicos e módulos, essa base permitirá avançar para visualizações, correlações, storytelling e análises exploratórias mais completas.
